# Bronze-ingest: SSB → Delta-tabeller

Henter rådata fra SSBs åpne API og lagrer som Delta-tabeller i Unity Catalog.

**Datakilder**

- SSB tabell 07459: Befolkning etter alder og kommune
- SSB tabell 11654: Lønnstakere og månedslønn per næring

**Hva notebooken gjør**

1. Oppretter katalog og schema i Unity Catalog
2. Definerer en enkel JSON-stat2-parser (~30 linjer)
3. Henter data fra SSB med `requests.post()`
4. Skriver til `pensjon_lakehouse.bronze.*` som Delta-tabeller

Ingen secrets, ingen avhengigheter utover `requests` og `pyspark`.

## 1. Opprett katalog og schema

In [0]:
spark.sql("CREATE CATALOG IF NOT EXISTS pensjon_lakehouse")
spark.sql("USE CATALOG pensjon_lakehouse")
spark.sql("CREATE SCHEMA IF NOT EXISTS bronze")

print("✓ Katalog og schema opprettet")

## 2. JSON-stat2-parser

SSB returnerer data i JSON-stat2-format: en flat verdiliste + dimensjonsbeskrivelser.
Parseren folder dem sammen til vanlige rader.

In [0]:
def parse_jsonstat2(response: dict) -> list[dict]:
    """Konverter JSON-stat2 response til en liste med rader."""
    dim_ids = response["id"]
    dim_sizes = response["size"]
    values = response["value"]

    # Bygg oppslag: dimensjon → [koder], dimensjon → {kode: label}
    dim_codes = {}
    dim_labels = {}
    for dim_id in dim_ids:
        cat = response["dimension"][dim_id]["category"]
        index = cat["index"]
        label = cat.get("label", {})
        codes = sorted(index.keys(), key=lambda k: index[k]) if isinstance(index, dict) else index
        dim_codes[dim_id] = codes
        dim_labels[dim_id] = {c: label.get(c, c) for c in codes}

    # Fold ut flat array til rader
    rows = []
    for flat_idx in range(len(values)):
        row = {}
        remainder = flat_idx
        for i, dim_id in enumerate(dim_ids):
            stride = 1
            for j in range(i + 1, len(dim_ids)):
                stride *= dim_sizes[j]
            dim_idx = remainder // stride
            remainder = remainder % stride
            code = dim_codes[dim_id][dim_idx]
            row[f"{dim_id}_code"] = code
            row[f"{dim_id}_label"] = dim_labels[dim_id][code]
        row["value"] = values[flat_idx]
        rows.append(row)

    return rows

print("✓ parse_jsonstat2 definert")

## 3. Hent befolkningsdata fra SSB (tabell 07459)

In [0]:
import requests

def fetch_befolkning(years: list[str] = None) -> dict:
    """Hent befolkning per alder og kommune fra SSB."""
    time_filter = (
        {"code": "Tid", "selection": {"filter": "item", "values": years}}
        if years
        else {"code": "Tid", "selection": {"filter": "top", "values": ["5"]}}
    )
    query = {
        "query": [
            {"code": "Region",       "selection": {"filter": "all", "values": ["*"]}},
            {"code": "Alder",        "selection": {"filter": "all", "values": ["*"]}},
            {"code": "ContentsCode", "selection": {"filter": "item", "values": ["Personer1"]}},
            time_filter,
        ],
        "response": {"format": "json-stat2"},
    }
    resp = requests.post("https://data.ssb.no/api/v0/no/table/07459", json=query, timeout=60)
    resp.raise_for_status()
    return resp.json()

raw_befolkning = fetch_befolkning()
rows_befolkning = parse_jsonstat2(raw_befolkning)
print(f"✓ Befolkning: {len(rows_befolkning)} rader hentet fra SSB")

## 4. Hent lønns-/sysselsettingsdata fra SSB (tabell 11654)

In [0]:
def fetch_lonn(quarters: list[str] = None) -> dict:
    """Hent lønnstakere og månedslønn per næring fra SSB."""
    time_filter = (
        {"code": "Tid", "selection": {"filter": "item", "values": quarters}}
        if quarters
        else {"code": "Tid", "selection": {"filter": "top", "values": ["4"]}}
    )
    query = {
        "query": [
            {"code": "NACE2007",     "selection": {"filter": "all", "values": ["*"]}},
            {"code": "ContentsCode", "selection": {"filter": "item", "values": ["Lonsstakere", "GjMdTotal"]}},
            time_filter,
        ],
        "response": {"format": "json-stat2"},
    }
    resp = requests.post("https://data.ssb.no/api/v0/no/table/11654", json=query, timeout=60)
    resp.raise_for_status()
    return resp.json()

raw_lonn = fetch_lonn()
rows_lonn = parse_jsonstat2(raw_lonn)
print(f"✓ Lønn/sysselsetting: {len(rows_lonn)} rader hentet fra SSB")

## 5. Skriv til Bronze Delta-tabeller

In [0]:
from pyspark.sql import functions as F
from datetime import datetime

batch_id = f"batch_{datetime.now().strftime('%Y%m%d_%H%M%S')}"

# --- Befolkning ---
df_bef = spark.createDataFrame(rows_befolkning)
df_bronze_bef = (
    df_bef
    .withColumn("_ingest_ts", F.current_timestamp())
    .withColumn("_source", F.lit("ssb_07459"))
    .withColumn("_batch_id", F.lit(batch_id))
)
df_bronze_bef.write.mode("overwrite").saveAsTable("pensjon_lakehouse.bronze.ssb_befolkning_raw")

# --- Lønn/sysselsetting ---
df_lonn = spark.createDataFrame(rows_lonn)
df_bronze_lonn = (
    df_lonn
    .withColumn("_ingest_ts", F.current_timestamp())
    .withColumn("_source", F.lit("ssb_11654"))
    .withColumn("_batch_id", F.lit(batch_id))
)
df_bronze_lonn.write.mode("overwrite").saveAsTable("pensjon_lakehouse.bronze.ssb_lonn_sysselsetting_raw")

count_bef = spark.table("pensjon_lakehouse.bronze.ssb_befolkning_raw").count()
count_lonn = spark.table("pensjon_lakehouse.bronze.ssb_lonn_sysselsetting_raw").count()

print(f"✓ Batch: {batch_id}")
print(f"✓ bronze.ssb_befolkning_raw: {count_bef} rader")
print(f"✓ bronze.ssb_lonn_sysselsetting_raw: {count_lonn} rader")

## 6. Verifiser

In [0]:
display(spark.table("pensjon_lakehouse.bronze.ssb_befolkning_raw").limit(5))

In [0]:
display(spark.table("pensjon_lakehouse.bronze.ssb_lonn_sysselsetting_raw").limit(5))

## Ferdig

Bronze-tabellene ligger nå i Unity Catalog som Delta-tabeller:

- `pensjon_lakehouse.bronze.ssb_befolkning_raw`
- `pensjon_lakehouse.bronze.ssb_lonn_sysselsetting_raw`

Kjør **02_silver_gold** for å bygge Silver- og Gold-lagene med ren SQL.